# ECOCANAL IA — guía para Google Colab

**Proyecto académico de predicción de riesgo de residuos en canales de agua.**

Esta libreta permite reproducir el dataset sintético, entrenar los dos modelos y verificar las métricas dentro de Google Colab. La aplicación Streamlit está diseñada para ejecutarse localmente; al final se incluye una alternativa para visualizarla desde Colab.

> Los datos son sintéticos y el sistema no realiza mediciones ambientales reales.


## 1. Preparar el proyecto

Sube el archivo `ECOCANAL_IA.zip` (o monta la carpeta del proyecto en Google Drive). Si usas el zip, ejecuta la siguiente celda. Si ya tienes la carpeta en `/content/ECOCANAL_IA`, puedes omitirla.


In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall('/content')

PROJECT_DIR = Path('/content/ECOCANAL_IA')
print('Proyecto disponible en:', PROJECT_DIR)


## 2. Instalar dependencias

Google Colab normalmente incluye TensorFlow. Instala el resto de dependencias del proyecto.


In [ ]:
%cd /content/ECOCANAL_IA
!pip -q install -r requirements.txt


## 3. Regenerar el dataset sintético

El dataset incluido ya está generado. Esta celda permite reproducirlo con la semilla documentada.


In [ ]:
!python -m src.generate_dataset


## 4. Entrenar y evaluar ambos modelos

Se guardan en `models/` los modelos, el escalador, los metadatos y las métricas de prueba.


In [ ]:
!python -m src.train_models


## 5. Verificación rápida de resultados


In [ ]:
import json

import pandas as pd

from src.predictions import load_artifacts, predict_single

with open('models/model_metadata.json', encoding='utf-8') as file:
    metadata = json.load(file)

rows = []
for key in ('logistica', 'red_neuronal'):
    item = metadata['models'][key]
    rows.append({
        'Modelo': item['label'],
        'Accuracy': item['accuracy'],
        'Precision': item['precision'],
        'Recall': item['recall'],
        'F1': item['f1'],
        'ROC-AUC': item['roc_auc'],
        'Tiempo (s)': item['training_time_seconds'],
    })
display(pd.DataFrame(rows))

artifacts = load_artifacts()
result = predict_single({
    'basura_acumulada': 82,
    'obstruccion': 88,
    'flujo_agua': 14,
    'frecuencia_lluvias': 26,
    'dias_sin_limpieza': 410,
    'residuos_plasticos': 3600,
}, artifacts)
print('Riesgo consensuado:', result['consenso']['nivel_riesgo'])
print('Probabilidad:', round(result['consenso']['probabilidad'], 4))
print('Recomendación:', result['recomendacion_principal'])


## 6. Ejecutar Streamlit en Colab (opcional)

La forma recomendada de usar la interfaz es localmente con `streamlit run app.py`. Si necesitas una demostración desde Colab, puedes abrir el puerto 8501 mediante el proxy de Colab. Es posible que el túnel tarde unos segundos o requiera volver a ejecutar la celda.


In [ ]:
import time

from google.colab.output import eval_js

!nohup streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /tmp/streamlit.log 2>&1 &
time.sleep(5)

print('Abre la aplicación en:')
print(eval_js('google.colab.kernel.proxyPort(8501)'))


## Notas

- Si TensorFlow no se instala, verifica la versión de Python (se recomienda 3.11 o 3.12).
- La app también funciona sin reentrenar si la carpeta `models/` contiene los artefactos.
- Consulta `README.md` para la documentación completa, limitaciones y comandos de Windows.
